In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('../src'))
from dataPreprocessing import load_and_preprocess_raw
from featureEngineering import generate_all_features

RAW_DIR = '../dataset/raw/'

# ==========================================
# BƯỚC 1 & 2: TIỀN XỬ LÝ (Lấy nhiều dữ liệu hơn)
# ==========================================
print("🚀 BẮT ĐẦU CHẠY PIPELINE DỮ LIỆU...")
# Nâng số dòng lên để lấy được nhiều sản phẩm (Báo cáo thật có thể bỏ nrows đi)
df_master = load_and_preprocess_raw(RAW_DIR, nrows=50000)
df_featured = generate_all_features(df_master, RAW_DIR)
print("\n✅ Pipeline Dữ liệu hoàn tất!\n")

# ==========================================
# BƯỚC 4: CHUẨN BỊ DỮ LIỆU GLOBAL (TOÀN BỘ DÒNG SẢN PHẨM)
# ==========================================
sample_store = df_featured['store_id'].iloc[0]
sample_dept = df_featured['dept_id'].iloc[0] # Lấy dòng sản phẩm (VD: FOODS_1, HOBBIES_1...)

print(f"🌍 Đang chuẩn bị dữ liệu Global cho TOÀN BỘ dòng sản phẩm: {sample_dept} tại {sample_store}")

# Lọc toàn bộ sản phẩm thuộc dòng này tại cửa hàng này
df_global = df_featured[(df_featured['store_id'] == sample_store) & (df_featured['dept_id'] == sample_dept)].copy()

# THUẬT TOÁN QUAN TRỌNG: Tính Lag và Rolling an toàn cho nhiều sản phẩm
print(f"-> Đang tính toán độ trễ (Lags) độc lập cho {df_global['item_id'].nunique()} mặt hàng...")
df_global = df_global.sort_values(['item_id', 'date'])
df_global['lag_7'] = df_global.groupby('item_id')['demand'].shift(7)
df_global['lag_14'] = df_global.groupby('item_id')['demand'].shift(14)
df_global['rolling_mean_7'] = df_global.groupby('item_id')['demand'].transform(lambda x: x.shift(1).rolling(7).mean())

df_global = df_global.dropna() # Xóa các dòng NaN do shift lag

# Đưa item_id vào làm Category để AI phân biệt được các món hàng
df_global['item_id'] = df_global['item_id'].astype('category')

features = [
    'item_id', 'day', 'is_weekend', 'sell_price', 'price_discount', 
    'event_name_1', 'lag_7', 'lag_14', 'rolling_mean_7'
]
target = 'demand'

# Tách tập Train/Test theo thời gian cho TOÀN BỘ sản phẩm
max_date = df_global['date'].max()
split_date = max_date - pd.Timedelta(days=28)

train_data_global = df_global[df_global['date'] <= split_date]
test_data_global = df_global[df_global['date'] > split_date]

X_train, y_train = train_data_global[features], train_data_global[target]

# ==========================================
# BƯỚC 5: HUẤN LUYỆN GLOBAL MODEL
# ==========================================
print("\n🧠 Đang huấn luyện AI Global (Học chéo - Cross Learning)...")
params = {
    'objective': 'tweedie',
    'tweedie_variance_power': 1.1,
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63, # Tăng số lá vì dữ liệu phức tạp hơn
    'verbose': -1,
    'force_col_wise': True
}

categorical_features = ['item_id', 'event_name_1']
train_set = lgb.Dataset(X_train, y_train, categorical_feature=categorical_features)

model = lgb.train(params, train_set, num_boost_round=150) 
print("✅ Huấn luyện AI hoàn tất!")

# ==========================================
# BƯỚC 6: CHỌN 1 SẢN PHẨM ĐỂ MÔ PHỎNG VÀ VẼ BIỂU ĐỒ
# (Dù AI học trên tất cả, nhưng nhập kho phải tính riêng từng món)
# ==========================================
target_item = df_global['item_id'].unique()[0] # Chọn món hàng đầu tiên làm mẫu
print(f"\n🎯 Trích xuất dự báo và chạy Mô phỏng Tồn kho riêng cho: {target_item}")

df_target_test = test_data_global[test_data_global['item_id'] == target_item].copy()
X_test_target = df_target_test[features]
y_actual_target = df_target_test[target].values

# AI dự báo trên tập test của sản phẩm mục tiêu
daily_demand_forecast = np.round(model.predict(X_test_target)).astype(int)

# Chạy Monte Carlo Simulation
LEAD_TIME = 3            
ORDERING_COST = 50.0     
HOLDING_COST = 0.5       
STOCKOUT_COST = 5.0      
INITIAL_INVENTORY = 15   

def simulate_inventory(demand_array, ROP, Q):
    inventory = INITIAL_INVENTORY
    total_cost = 0
    days_to_arrival = 0
    order_placed = False
    inventory_history = []
    for demand in demand_array:
        if order_placed and days_to_arrival == 0:
            inventory += Q
            order_placed = False
        if inventory >= demand:
            inventory -= demand
        else:
            total_cost += (demand - inventory) * STOCKOUT_COST
            inventory = 0
        total_cost += inventory * HOLDING_COST
        if inventory <= ROP and not order_placed:
            total_cost += ORDERING_COST
            order_placed = True
            days_to_arrival = LEAD_TIME
        if order_placed:
            days_to_arrival -= 1
        inventory_history.append(inventory)
    return total_cost, inventory_history

best_cost, best_ROP, best_Q = float('inf'), 0, 0
best_history = []

for rop in range(0, 50, 2):
    for q in range(10, 150, 5):
        cost, history = simulate_inventory(daily_demand_forecast, rop, q)
        if cost < best_cost:
            best_cost, best_ROP, best_Q, best_history = cost, rop, q, history

# ==========================================
# BÁO CÁO VÀ VẼ BIỂU ĐỒ
# ==========================================
print("\n" + "="*60)
print(f"📊 BÁO CÁO SAI SỐ (MÔ HÌNH GLOBAL) - MÃ {target_item} 📊")
print("="*60)
mae = mean_absolute_error(y_actual_target, daily_demand_forecast)
rmse = np.sqrt(mean_squared_error(y_actual_target, daily_demand_forecast))
print(f"MAE: {mae:,.2f} | RMSE: {rmse:,.2f}")

daily_errors = y_actual_target - daily_demand_forecast

# Vẽ biểu đồ 3 tầng
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 18), facecolor='#FAFAFA')

ax1.bar(range(1, 29), daily_demand_forecast, color='#3498DB', edgecolor='black', alpha=0.6, label='Dự báo AI Global')
ax1.plot(range(1, 29), y_actual_target, color='#D35400', marker='o', linewidth=2.5, label='Nhu cầu Thực tế')
ax1.set_title(f"PHẦN 1: SO SÁNH DỰ BÁO VS THỰC TẾ - {target_item}", fontsize=14, fontweight='bold', pad=10)
ax1.set_ylabel("Số lượng bán", fontsize=12, fontweight='bold')
ax1.set_xticks(range(1, 29))
ax1.grid(axis='y', linestyle='--', alpha=0.5)
ax1.legend(loc='upper left')

colors = ['#E74C3C' if e > 0 else '#2ECC71' for e in daily_errors]
ax2.bar(range(1, 29), daily_errors, color=colors, edgecolor='black', alpha=0.8)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=2)
ax2.set_title(f"PHẦN 2: LỖI SAI LỆCH HÀNG NGÀY", fontsize=14, fontweight='bold', pad=10)
ax2.set_ylabel("Lỗi (sp)", fontsize=12, fontweight='bold')
ax2.set_xticks(range(1, 29))
ax2.grid(axis='y', linestyle='--', alpha=0.5)

ax3.step(range(1, 29), best_history, where='post', color='#27AE60', linewidth=2.5, label='Lượng Tồn Kho (Mô phỏng)')
ax3.axhline(y=best_ROP, color='#E74C3C', linestyle='--', linewidth=2, label=f'Ngưỡng ROP = {best_ROP}')
ax3.set_title(f"PHẦN 3: MÔ PHỎNG TỒN KHO TỐI ƯU (EOQ = {best_Q})", fontsize=14, fontweight='bold', pad=10)
ax3.set_xlabel("Ngày dự báo", fontsize=12, fontweight='bold')
ax3.set_ylabel("Số lượng trong kho", fontsize=12, fontweight='bold')
ax3.set_xticks(range(1, 29))
ax3.grid(True, linestyle='-.', alpha=0.5)
ax3.legend(loc='upper right')

plt.tight_layout(pad=3.0) 
os.makedirs('../docs/images', exist_ok=True)
plt.savefig('../docs/images/bieu_do_global_model.png', dpi=300, bbox_inches='tight')
print("\n📸 Đã lưu ảnh thành công!")
plt.show()